In [21]:
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
import os
from azure.search.documents.indexes.models import SearchIndex, SearchField, VectorSearch, VectorSearchProfile, HnswAlgorithmConfiguration, AzureOpenAIVectorizer, AzureOpenAIVectorizerParameters, SemanticSearch, SemanticConfiguration, SemanticPrioritizedFields, SemanticField
from azure.search.documents.indexes import SearchIndexClient
import requests
from azure.search.documents import SearchIndexingBufferedSender
from azure.search.documents.indexes.models import KnowledgeAgent, KnowledgeAgentAzureOpenAIModel, KnowledgeAgentTargetIndex, KnowledgeAgentRequestLimits, AzureOpenAIVectorizerParameters

In [22]:


load_dotenv(override=True) # take environment variables from .env.


project_endpoint = os.environ["PROJECT_ENDPOINT"]
agent_model = os.getenv("AGENT_MODEL", "gpt-4.1-mini")
endpoint = os.environ["AZURE_SEARCH_ENDPOINT"]
credential = DefaultAzureCredential(managed_identity_client_id  = "fff597f2-4818-46d2-a58a-c4e105847b1a")
token_provider = get_bearer_token_provider(credential, "https://search.azure.com/.default")
index_name = os.getenv("AZURE_SEARCH_INDEX", "agenticai")
azure_openai_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
azure_openai_gpt_deployment = os.getenv("AZURE_OPENAI_GPT_DEPLOYMENT", "gpt-4.1-mini")
azure_openai_gpt_model = os.getenv("AZURE_OPENAI_GPT_MODEL", "gpt-4.1-mini")
azure_openai_embedding_deployment = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "text-embedding-3-large")
azure_openai_embedding_model = os.getenv("AZURE_OPENAI_EMBEDDING_MODEL", "text-embedding-3-large")
agent_name = os.getenv("AZURE_SEARCH_AGENT_NAME", "earth-search-agent")

In [23]:
print(credential)

In [24]:
index = SearchIndex(
    name=index_name,
    fields=[
        # Key field
        SearchField(name="id", type="Edm.String", key=True, filterable=True, sortable=False, facetable=False, searchable=False),
        
        # String fields
        SearchField(name="dataset", type="Edm.String", filterable=True, sortable=True, facetable=True, searchable=True, analyzer_name="standard"),
        SearchField(name="domain", type="Edm.String", filterable=True, sortable=True, facetable=True, searchable=True, analyzer_name="standard"),
        SearchField(name="source_kind", type="Edm.String", filterable=True, sortable=True, facetable=True, searchable=True, analyzer_name="standard"),
        SearchField(name="media_type", type="Edm.String", filterable=True, sortable=True, facetable=True, searchable=True, analyzer_name="standard"),
        SearchField(name="authority", type="Edm.String", filterable=True, sortable=True, facetable=True, searchable=True, analyzer_name="standard"),
        SearchField(name="heading", type="Edm.String", filterable=True, sortable=True, facetable=True, searchable=True, analyzer_name="standard"),
        SearchField(name="link", type="Edm.String", filterable=True, sortable=True, facetable=True, searchable=True, analyzer_name="standard"),
        SearchField(name="content", type="Edm.String", filterable=False, sortable=False, facetable=False, searchable=True, analyzer_name="standard"),
        
        # Boolean field
        SearchField(name="is_latest", type="Edm.Boolean", filterable=True, sortable=True, facetable=True, searchable=False),
        
        # DateTime field
        SearchField(name="date", type="Edm.DateTimeOffset", filterable=True, sortable=True, facetable=True, searchable=False),
        
        # Vector field
        SearchField(name="content_vector", type="Collection(Edm.Single)", stored=False, vector_search_dimensions=3072, vector_search_profile_name="hnsw_text_3_large"),
        
        # Complex collection field with k and v sub-fields
        SearchField(
            name="additional_kv", 
            type="Collection(Edm.ComplexType)",
            fields=[
                SearchField(name="k", type="Edm.String", filterable=True, facetable=True, searchable=True, analyzer_name="standard"),
                SearchField(name="v", type="Edm.String", filterable=True, facetable=True, searchable=True, analyzer_name="standard")
            ]
        )
    ],
    vector_search=VectorSearch(
        profiles=[VectorSearchProfile(name="hnsw_text_3_large", algorithm_configuration_name="alg", vectorizer_name="azure_openai_text_3_large")],
        algorithms=[HnswAlgorithmConfiguration(name="alg")],
        vectorizers=[
            AzureOpenAIVectorizer(
                vectorizer_name="azure_openai_text_3_large",
                parameters=AzureOpenAIVectorizerParameters(
                    resource_url=azure_openai_endpoint,
                    deployment_name=azure_openai_embedding_deployment,
                    model_name=azure_openai_embedding_model
                )
            )
        ]
    ),
    semantic_search=SemanticSearch(
        default_configuration_name="semantic_config",
        configurations=[
            SemanticConfiguration(
                name="semantic_config",
                prioritized_fields=SemanticPrioritizedFields(
                    content_fields=[
                        SemanticField(field_name="content")
                    ]
                )
            )
        ]
    )
)

index_client = SearchIndexClient(endpoint=endpoint, credential=credential)
index_client.create_or_update_index(index)
print(f"Index '{index_name}' created or updated successfully")

Index 'agenticai' created or updated successfully


In [25]:


# url = "https://raw.githubusercontent.com/Azure-Samples/azure-search-sample-data/refs/heads/main/nasa-e-book/earth-at-night-json/documents.json"
# documents = requests.get(url).json()
# print(len(documents))

In [26]:
# documents

In [27]:

# with SearchIndexingBufferedSender(endpoint=endpoint, index_name=index_name, credential=credential) as client:
#     client.upload_documents(documents=documents)

# print(f"Documents uploaded to index '{index_name}'")

In [28]:
# agent = KnowledgeAgent(
#     name=agent_name,
#     models=[
#         KnowledgeAgentAzureOpenAIModel(
#             azure_open_ai_parameters=AzureOpenAIVectorizerParameters(
#                 resource_url=azure_openai_endpoint,
#                 deployment_name=azure_openai_gpt_deployment,
#                 model_name=azure_openai_gpt_model
#             )
#         )
#     ],
#     target_indexes=[
#         KnowledgeAgentTargetIndex(
#             index_name=index_name,
#             default_reranker_threshold=2.5
#         )
#     ],
#     request_limits=KnowledgeAgentRequestLimits(
#         max_output_size=10000
#     )
# )

# index_client = SearchIndexClient(endpoint=endpoint, credential=credential)
# index_client.create_or_update_agent(agent)
# print(f"Knowledge agent '{agent_name}' created or updated successfully")